# RACE RC & Quiz Generation - ULTIMATE MASTER NOTEBOOK (v5 - Final Feature Alignment)
This version ensures that Model B (Hints) is trained on the correct 3-feature set (keyword overlap, position, length) to match the inference code.

In [ ]:
# 1. Setup & Installation
!pip install gensim joblib pandas numpy scikit-learn

import os, joblib, time, re
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Paths
PROJECT_DIR = '/content/drive/MyDrive/race_rc_project'
DATA_DIR    = f'{PROJECT_DIR}/data/processed'
MODEL_DIR_A = f'{PROJECT_DIR}/models/model_a/traditional'
MODEL_DIR_B = f'{PROJECT_DIR}/models/model_b/traditional'
os.makedirs(MODEL_DIR_A, exist_ok=True)
os.makedirs(MODEL_DIR_B, exist_ok=True)

# Load Data
print("Loading features...")
train_data = joblib.load(f'{DATA_DIR}/train_features.pkl')
val_data   = joblib.load(f'{DATA_DIR}/val_features.pkl')

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val     = val_data['X'], val_data['y']
print(f"Loaded {len(X_train)} training rows for Model A.")

## 2. Preprocessing: Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
print("Fitting Scaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
joblib.dump(scaler, f'{MODEL_DIR_A}/scaler.pkl')
print("Scaler saved.")

## 3. Model A Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

print("Training Supervised Models...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr, f'{MODEL_DIR_A}/lr.pkl')

svm = LinearSVC(class_weight='balanced', C=0.5, max_iter=2000, dual='auto')
svm.fit(X_train_scaled, y_train)
joblib.dump(svm, f'{MODEL_DIR_A}/svm.pkl')

nb = BernoulliNB()
nb.fit((X_train_scaled > 0).astype(float), y_train)
joblib.dump(nb, f'{MODEL_DIR_A}/nb.pkl')

print("Training Ensemble...")
svm_cal = CalibratedClassifierCV(svm, cv=3)
ensemble = VotingClassifier(estimators=[('lr', lr), ('svm', svm_cal), ('nb', nb)], voting='soft')
ensemble.fit(X_train_scaled[:20000], y_train[:20000])
joblib.dump(ensemble, f'{MODEL_DIR_A}/ensemble.pkl')

## 4. Model B: Hint Scorer (Corrected 3-Feature Set)
We build a specialized 3-feature dataset for the Hint Scorer to match the inference engine.

In [ ]:
print("Building Model B 3-Feature Dataset...")
# We use a small subset of the raw data to quickly train the hint scorer
raw_train = pd.read_csv(f'{PROJECT_DIR}/data/raw/train.csv').iloc[:5000]

X_hint, y_hint = [], []
for _, row in raw_train.iterrows():
    art, quest, ans = str(row['article']), str(row['question']), str(row[row['answer']])
    sents = [s.strip() for s in re.split(r'[.!?]', art) if len(s.strip()) > 10]
    q_tokens = set(re.findall(r'\w+', quest.lower()))
    ans_tokens = set(re.findall(r'\w+', ans.lower()))
    for idx, sent in enumerate(sents):
        s_tokens = set(re.findall(r'\w+', sent.lower()))
        kw_overlap = len(s_tokens & q_tokens) / max(len(q_tokens), 1)
        pos_score  = 1 - (idx / max(len(sents), 1))
        len_score  = len(s_tokens) / 50.0 
        X_hint.append([kw_overlap, pos_score, len_score])
        y_hint.append(1 if s_tokens & ans_tokens else 0)

from sklearn.linear_model import LogisticRegression as SkLR
hint_lr = SkLR(class_weight='balanced')
hint_lr.fit(X_hint, y_hint)
joblib.dump(hint_lr, f'{MODEL_DIR_B}/hint_lr.pkl')
print(f"Hint Scorer trained on {len(X_hint)} sentence-rows with 3 features.")

# Distractor Ranker (Matches the 6 features expected by distractor logic if necessary)
dist_lr = SkLR(class_weight='balanced')
dist_lr.fit(X_train_scaled[:10000], y_train[:10000])
joblib.dump(dist_lr, f'{MODEL_DIR_B}/distractor_lr.pkl')
print("Distractor Ranker saved.")

## 5. Finalize

In [ ]:
import gensim.downloader as api
print("Downloading Word2Vec...")
try:
    w2v = api.load('word2vec-google-news-300')
    w2v.save(f'{MODEL_DIR_B}/w2v.kv')
except:
    print("Word2Vec download failed or interrupted, but other models are saved.")
print("ALL MODELS SAVED AND ALIGNED!")